# RestoreAI — Google Colab GPU test
Run **Runtime → Change runtime type → T4 GPU** first. Run cells in order. This notebook is for temporary development only. It exposes a public, unauthenticated API URL while the tunnel is active; **do not share the URL or upload sensitive photos**. Close the tunnel when done.

In [ ]:
import os, sys, subprocess, torch
print('Python:', sys.version)
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
assert torch.cuda.is_available(), 'Choose a GPU runtime in Colab first'
print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
import subprocess, pathlib
ROOT = pathlib.Path('/content/RestoreAI')
if not ROOT.exists():
    subprocess.run(['git','clone','--depth','1','--branch','feature/bilingual-ui','https://github.com/khuloodakram/RestoreAI.git',str(ROOT)],check=True)
else:
    subprocess.run(['git','-C',str(ROOT),'pull','--ff-only'],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','-r',str(ROOT/'server/requirements.txt'),'numpy<2','tqdm','opencv-python-headless','scipy','lmdb','pyyaml','addict','future','yapf'],check=True)

## Install BasicSR
BasicSR may fail to build on newer Colab Python versions. Apply the known packaging workaround. If imports fail due to a PyTorch/torchvision mismatch, use a fresh runtime with compatible versions rather than forcing unrelated downgrades.

In [ ]:
BASIC = pathlib.Path('/content/BasicSR')
if not BASIC.exists():
    subprocess.run(['git','clone','--depth','1','https://github.com/XPixelGroup/BasicSR.git',str(BASIC)],check=True)
setup = BASIC/'setup.py'
s = setup.read_text()
s = s.replace("return locals()['__version__']", "return '1.4.2'")
setup.write_text(s)
subprocess.run([sys.executable,'-m','pip','install','-q','--no-build-isolation','--no-deps',str(BASIC)],check=True)
from basicsr.archs.rrdbnet_arch import RRDBNet
print('BasicSR RRDBNet import OK')

In [ ]:
import urllib.request
weights_dir = pathlib.Path('/content/Real-ESRGAN/weights')
weights_dir.mkdir(parents=True,exist_ok=True)
weights = weights_dir/'RealESRGAN_x4plus.pth'
if not weights.exists():
    urllib.request.urlretrieve('https://github.com/xinntao/Real-ESRGAN/releases/download/v0.1.0/RealESRGAN_x4plus.pth',weights)
os.environ['RESTOREAI_WEIGHTS']=str(weights)
print('Checkpoint:', weights, weights.stat().st_size, 'bytes')

## Run real CUDA inference test
This calls the FastAPI endpoint locally and verifies the returned PNG dimensions. It does **not** prove that scratches or faces are restored.

In [ ]:
import os
os.chdir(ROOT)
subprocess.run([sys.executable,'-m','server.smoke_test_gpu'],check=True)

## Start the temporary API and HTTPS tunnel
The tunnel URL is public and unauthenticated. Use only test photos, do not distribute it, and stop the Colab runtime when finished.

In [ ]:
import time, requests
server_log=open('/content/restoreai-server.log','w')
server_proc=subprocess.Popen([sys.executable,'-m','uvicorn','server.app:app','--host','127.0.0.1','--port','8000'],cwd=str(ROOT),env=os.environ.copy(),stdout=server_log,stderr=subprocess.STDOUT)
for attempt in range(30):
    try:
        response=requests.get('http://127.0.0.1:8000/health',timeout=2)
        if response.ok: break
    except requests.RequestException: pass
    time.sleep(1)
else: raise RuntimeError('API did not start; inspect /content/restoreai-server.log')
print(response.json())

In [ ]:
import re, urllib.request
cloudflared=pathlib.Path('/content/cloudflared')
if not cloudflared.exists():
    urllib.request.urlretrieve('https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64',cloudflared)
    cloudflared.chmod(0o755)
tunnel_log=open('/content/restoreai-tunnel.log','w')
tunnel_proc=subprocess.Popen([str(cloudflared),'tunnel','--url','http://127.0.0.1:8000','--no-autoupdate'],stdout=tunnel_log,stderr=subprocess.STDOUT)
url=None
for _ in range(40):
    time.sleep(1)
    log=pathlib.Path('/content/restoreai-tunnel.log').read_text()
    match=re.search(r'https://[a-z0-9-]+\.trycloudflare\.com',log)
    if match:
        url=match.group(0)
        break
if not url: raise RuntimeError('Tunnel did not start; inspect /content/restoreai-tunnel.log')
print('TEMPORARY HTTPS URL:',url)
print('Health:',url+'/health')
print('Android build flag: --dart-define=RESTOREAI_API_URL='+url)
print('This URL expires when Colab stops. Keep it private.')

## Stop services
Run the cell below to shut down the public tunnel and API.

In [ ]:
for proc in (globals().get('tunnel_proc'),globals().get('server_proc')):
    if proc is not None and proc.poll() is None:
        proc.terminate()
print('Stopped RestoreAI test services')